In [0]:
-- WHAT THIS QUERY DOES
-- We have two lists of companies:
--   List A: companies where a sanctioned owner REALLY holds half or more (multiplying, 2B)
--   List B: companies the 50% rule BLOCKED (2A)
-- We put the two lists side by side and count three groups:
--   in both lists / only in List B / only in List A

-- STEP 1: build List A
-- Take every owner -> company pair.
-- Keep it only if:
--   the owner is sanctioned,
--   the company is NOT sanctioned itself,
--   and the owner really holds half or more.
-- DISTINCT: count each company once, even if two sanctioned owners reach it.
WITH eff50 AS (
  SELECT DISTINCT e.asset_id AS id
  FROM pep_dev.eval.eff_ownership AS e
  JOIN pep_dev.eval.seeds AS s
    ON e.top_id = s.node_id          -- the owner is sanctioned
  LEFT ANTI JOIN pep_dev.eval.seeds AS s2
    ON e.asset_id = s2.node_id       -- the company is NOT sanctioned itself
  WHERE e.eff_share >= 0.5           -- the owner holds half or more
),

-- STEP 2: build List B
-- Take the companies the 50% rule blocked in 2A.
-- round > 0 skips round 0, which is just the sanctioned owners themselves.
ofac AS (
  SELECT node_id AS id
  FROM pep_dev.eval.blocked
  WHERE round > 0
)

-- STEP 3: put the lists side by side and count
-- FULL OUTER JOIN keeps every company from BOTH lists.
-- If a company is missing from one list, that side shows NULL (empty).
--   In both lists        ->  x.id filled, o.id filled
--   Only in List B       ->  x.id empty
--   Only in List A       ->  o.id empty
SELECT
  count_if(x.id IS NOT NULL AND o.id IS NOT NULL) AS both,
  count_if(x.id IS NULL)                          AS blocked_only,
  count_if(o.id IS NULL)                          AS effective_only
FROM eff50 AS x
FULL OUTER JOIN ofac AS o
  ON x.id = o.id

-- RESULT: 10,387 both / 704 blocked only / 9 effective only
-- CHECK:  10,387 + 704 = 11,091 (List B)   and   10,387 + 9 = 10,396 (List A)

In [0]:
-- WHAT THIS QUERY DOES
-- Shows the 9 companies where a sanctioned owner really holds half or more,
-- but the 50% rule did NOT block them. Then we check they're real.

-- STEP 1: the companies the 50% rule blocked (same as List B above)
WITH ofac AS (
  SELECT node_id FROM pep_dev.eval.blocked WHERE round > 0
)

-- STEP 2: find the companies, keeping a row only if:
--   the owner is sanctioned,
--   the company is NOT sanctioned itself,
--   the company was NOT blocked by the 50% rule,
--   and the owner holds half or more.
-- The last join just adds the company's name.
SELECT e.top_id, e.asset_id, le.name AS company,
       e.eff_share, e.n_paths, e.min_depth, e.max_depth, e.has_conflict
FROM pep_dev.eval.eff_ownership AS e
JOIN pep_dev.eval.seeds AS s
  ON e.top_id = s.node_id            -- the owner is sanctioned
LEFT ANTI JOIN pep_dev.eval.seeds AS s2
  ON e.asset_id = s2.node_id         -- the company is NOT sanctioned itself
LEFT ANTI JOIN ofac AS o
  ON e.asset_id = o.node_id          -- the 50% rule did NOT block it
LEFT JOIN pep_dev.silver.legal_entity AS le
  ON le.canonical_id = e.asset_id    -- add the company's name
WHERE e.eff_share >= 0.5             -- the owner holds half or more
ORDER BY e.eff_share DESC

-- RESULT: 9 companies, all with has_conflict = false, so they're real.